# Topic 14: PyTorch essentials

CSCI 3320 · Neural Networks II · tensors, autograd, the training loop

This notebook is the code from the [Topic 14 lecture note](https://dongchul.kim/csci3320-fall2026/topic/13), in the same order.
Run the cells from top to bottom (Runtime > Run all works too), compare what you see
with the note, then change things in the **Try it** cells.

It runs on a CPU in a few seconds; no GPU is needed. The printed numbers can differ a
little from the note on a different PyTorch version.

## 1. Tensors: NumPy arrays that remember

In [ ]:
import numpy as np
import torch

t = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
print(t.shape, t.dtype)
print(t * 10)
print(t.sum(), t[1, 2])

a = np.array([1.0, 2.0])
print(torch.tensor(a), torch.tensor(a).numpy())

A NumPy array of decimals becomes a `float64` tensor, but PyTorch's layers expect `float32`.
That is why the program in section 3 writes `dtype=torch.float32`.

## 2. Autograd: PyTorch finds the slope for you

Topic 8a's hand example: `x = [1, 2]`, `y = [2, 4]`, the model `w * x`, starting at `w = 0`.
By hand you found the cost 5 and the slope -5.

In [ ]:
import torch

x = torch.tensor([1.0, 2.0])
y = torch.tensor([2.0, 4.0])
w = torch.tensor(0.0, requires_grad=True)   # a knob: PyTorch, please track it

cost = ((w * x - y) ** 2).mean() / 2        # Topic 8a's cost, at w = 0
print(cost)
cost.backward()                             # work out the slope for every tracked knob
print(w.grad)

### Gradients add up

Run the cost and `.backward()` again without changing `w`. The slope is still -5, but
`w.grad` now says -10: PyTorch **adds** each new gradient to `.grad`. That is why every
training loop starts with `optimizer.zero_grad()`.

In [ ]:
cost = ((w * x - y) ** 2).mean() / 2        # same cost again, w has not moved
cost.backward()
print(w.grad)                               # added to the old one, not replaced

**Try it.** Run the cell above a few more times. What does `w.grad` say each time?
Then run `w.grad.zero_()` in a new cell and run the cell above once more.

## 3. The whole program: iris 4-8-3

The five steps per batch: **zero_grad, forward, loss, backward, step**.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
X, y = load_iris(return_X_y=True)
X, y = torch.tensor(X, dtype=torch.float32), torch.tensor(y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
loader = DataLoader(TensorDataset(X_train, y_train), batch_size=20, shuffle=True)

model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 3))
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.02)

for epoch in range(100):
    for xb, yb in loader:
        optimizer.zero_grad()          # 1. clear old gradients
        pred = model(xb)               # 2. forward: make predictions
        loss = loss_fn(pred, yb)       # 3. measure how wrong
        loss.backward()                # 4. backward: find every gradient
        optimizer.step()               # 5. step: nudge every weight
    if (epoch + 1) % 20 == 0:
        print(f"epoch {epoch + 1:3d}   loss {loss.item():.4f}")

model.eval()
with torch.no_grad():
    acc = (model(X_test).argmax(dim=1) == y_test).float().mean()
print(f"test accuracy: {acc.item():.4f}")

The note's run prints a loss of about 0.85 after 20 epochs, 0.22 after 100, and a test
accuracy of 1.0000 (30 of 30 flowers).

**Try it.** Change `lr=0.02` to `0.002` and to `0.5`, run the cell again each time, and
watch the loss. Then put it back.

## 4. When a step is missing or out of order

The same program, with one thing broken at a time. Every version uses the same seed and
the same data, so only the mistake differs. PyTorch raises **no error** for any of them.

In [ ]:
def run(variant):
    torch.manual_seed(0)
    X, y = load_iris(return_X_y=True)
    X, y = torch.tensor(X, dtype=torch.float32), torch.tensor(y)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
    loader = DataLoader(TensorDataset(X_train, y_train), batch_size=20, shuffle=True)
    model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 3))
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.02)
    losses = []
    for epoch in range(100):
        for xb, yb in loader:
            if variant != "no zero_grad":
                optimizer.zero_grad()
            if variant == "step before backward":
                optimizer.step()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            if variant != "no backward":
                loss.backward()
            if variant not in ("step before backward", "no step"):
                optimizer.step()
        if (epoch + 1) % 20 == 0:
            losses.append(round(loss.item(), 2))
    model.eval()
    with torch.no_grad():
        acc = (model(X_test).argmax(dim=1) == y_test).float().mean().item()
    print(f"{variant:22s} loss {losses}  test accuracy {acc:.4f}")


for v in ["correct", "no zero_grad", "step before backward", "no backward", "no step"]:
    run(v)

Three of the four mistakes print **identical** numbers: in all three the weights never
change, the loss stays near 1.2, and the network guesses at chance (1 in 3).
A missing `zero_grad()` makes the loss jump around instead.

## 5. Testing: `model.eval()` and `torch.no_grad()`

Outside `no_grad()` an output carries a record (`grad_fn`); inside it, there is none.

In [ ]:
torch.manual_seed(0)
m = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 3))
x = torch.rand(2, 4)
print("without no_grad:", m(x).grad_fn)
with torch.no_grad():
    print("with no_grad:   ", m(x).grad_fn)

## What to take away

Every PyTorch training loop is the same five lines in the same order:
**zero_grad, forward, loss, backward, step**. A flat loss means the weights are not
moving; a wild loss often means gradients are piling up. Back to the
[lecture note](https://dongchul.kim/csci3320-fall2026/topic/13).